# Mass Spectrometry Analysis of Extracellular Vesicles Isolated from Stimulated Human Mast Cells Exploration with `mlcroissant`
This notebook provides a template for loading and exploring a dataset using the `mlcroissant` library.

### Dataset Source
The dataset source is provided via a Croissant schema URL.

In [ ]:
# Ensure `mlcroissant` library is installed
!pip install mlcroissant

## 1. Data Loading
Load metadata and records from the dataset using `mlcroissant`.

In [ ]:
import mlcroissant as mlc
import pandas as pd

# Define the dataset URL
croissant_url = 'https://sen.science/doi/10.71728/senscience.vq4a-28xa/fair2.json'

# Load the dataset metadata
dataset = mlc.Dataset(croissant_url)
# Access the dataset.metadata object (not dict), then convert to dict for display
meta = dataset.metadata.to_json()
print(f"{meta['name']}: {meta['description']}")

## 2. Data Overview
Review available record sets, fields (columns), and their `@id`s.

We will list each record set with its `@id`, the file it's based on, and its field (column) `@id`s.

In [ ]:
# List all available record sets and their fields
record_sets = dataset.metadata.record_sets
if not record_sets:
    print("No record sets found in the dataset.")
else:
    for rs in record_sets:
        print(f"RecordSet @id: {rs['@id']}")
        if 'name' in rs:
            print(f"  Name: {rs['name']}")
        if 'description' in rs:
            print(f"  Description: {rs['description']}")
        file_object_ids = rs.get('fileObject', [])
        if file_object_ids:
            if isinstance(file_object_ids, list):
                for fo in file_object_ids:
                    print(f"  FileObject @id: {fo}")
            else:
                print(f"  FileObject @id: {file_object_ids}")
        field_ids = rs.get('field', [])
        if field_ids:
            print(f"  Fields: {field_ids}")

## 3. Data Extraction
Load data from one or more record sets into DataFrames for analysis. Use the record set and field `@id`s from above.

> **Note:** All IDs are referenced by their `@id` as required.

In [ ]:
# Discover record set IDs
rs_ids = [rs['@id'] for rs in dataset.metadata.record_sets]
print("Record Set IDs:", rs_ids)

# Extract data from each record set
# For demonstration, we load all record sets (usually just one in most Croissant datasets).
dataframes = {}

for record_set_id in rs_ids:
    records = list(dataset.records(record_set=record_set_id))
    df = pd.DataFrame(records)
    dataframes[record_set_id] = df
    print(f"Data loaded for RecordSet @id: {record_set_id}")
    print(f"Columns in this set:", df.columns.tolist())
    display(df.head())

## 4. Exploratory Data Analysis (EDA)
Apply common data processing steps, such as filtering records based on specific criteria, normalizing numeric fields, and grouping data. Some field `@id`s (column names) are shown above.

Let's select a record set and a numeric field for demonstration.

In [ ]:
# Select a record set and numeric field for analysis
if rs_ids:
    record_set_id = rs_ids[0]  # Select the first record set
    df = dataframes[record_set_id]
    print(f"Available columns: {df.columns.tolist()}")
    # Try to select a likely numeric field
    numeric_candidates = [c for c in df.columns if any(s in c.lower() for s in ["count", "abundance", "mw", "weight", "coverage", "value", "intensity"])]
    if numeric_candidates:
        numeric_field = numeric_candidates[0]
        print(f"Selected numeric field: {numeric_field}")
    else:
        numeric_field = df.select_dtypes(include='number').columns[0] if not df.select_dtypes(include='number').empty else df.columns[0]
        print(f"Selected field: {numeric_field}")
    
    # Try to select a group field
    group_candidates = [c for c in df.columns if any(s in c.lower() for s in ["accession", "description", "category", "type", "group", "protein"])]
    group_field = group_candidates[0] if group_candidates else df.columns[0]

    # Filter records on numeric field
    try:
        df[numeric_field] = pd.to_numeric(df[numeric_field], errors='coerce')
        threshold = df[numeric_field].mean()  # Use mean as threshold for demo
        filtered_df = df[df[numeric_field] > threshold].copy()
        print(f"Filtered records with {numeric_field} > {threshold}:")
        display(filtered_df.head())
        
        # Normalize numeric field
        filtered_df[f"{numeric_field}_normalized"] = (
            (filtered_df[numeric_field] - filtered_df[numeric_field].mean()) /
            filtered_df[numeric_field].std()
        )
        print(f"Normalized {numeric_field} for filtered records:")
        display(filtered_df[[numeric_field, f"{numeric_field}_normalized"]].head())
        
        # Group by group_field
        if group_field in filtered_df.columns:
            grouped_df = filtered_df.groupby(group_field)[numeric_field].mean().reset_index()
            print(f"Grouped mean {numeric_field} by {group_field}:")
            display(grouped_df.head())
        else:
            print(f"Group field '{group_field}' not found in DataFrame columns.")
    except Exception as e:
        print("Could not process numeric field for EDA. Error:", e)
else:
    print("No record set with tabular data found.")

## 5. Visualization
Visualize data distributions or the relationship between fields of interest. We'll plot a histogram of the selected numeric field and a bar plot for grouped means.

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns
%matplotlib inline

if rs_ids and numeric_field in df.columns:
    plt.figure(figsize=(8,4))
    sns.histplot(df[numeric_field].dropna(), bins=30, kde=True)
    plt.title(f'Distribution of {numeric_field}')
    plt.xlabel(numeric_field)
    plt.show()
    
    # Barplot for grouped means
    if 'grouped_df' in locals() and not grouped_df.empty and grouped_df.shape[0] < 50:
        plt.figure(figsize=(10,4))
        sns.barplot(x=group_field, y=numeric_field, data=grouped_df, ci=None)
        plt.title(f'Mean {numeric_field} by {group_field}')
        plt.xticks(rotation=45, ha='right')
        plt.show()
else:
    print("No numeric or suitable group data available for visualization.")

## 6. Conclusion
We have demonstrated how to access and explore the FAIR<sup>2</sup> protein abundance dataset using `mlcroissant`, referencing all entities by their `@id`, and walking through dynamic record set exploration, basic filtering, normalization, grouping, and plotting. 

The notebook can be reused for any Croissant-based dataset by substituting the schema URL, and all entity references remain consistent and robust via their IDs.